# Global Superstore – Profitability & Customer Segmentation Analysis

End-to-end exploratory data analysis using **Python and Pandas** to identify profitability drivers, loss-making segments, and customer targeting strategies.

## Business Objective

A multinational retail company operating across 147 countries wants to understand why margins are under pressure despite strong sales volume.  
This notebook investigates:

- which segments and categories generate the most value,
- where discounting starts to damage profitability,
- which regions underperform,
- and what actions leadership could take to improve margins.

> This notebook is intentionally written as a portfolio case study: clear, reproducible, and business-oriented.


## 1. Imports and setup

We will use:

- **Pandas** for data manipulation
- **NumPy** for numerical operations
- **Matplotlib / Seaborn** for visualizations


In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


## 2. Load the dataset

The `Orders` sheet contains **51,290 transaction rows (order line items)** from 2011 to 2014.


In [ ]:
# Load the Excel file
file_path = "Global Superstore.xls"
sheet_name = "Orders"

df = pd.read_excel(file_path, sheet_name=sheet_name)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()


## 3. Initial inspection

Before cleaning anything, let's inspect structure, datatypes, and missing values.


In [ ]:
df.info()


In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)


## 4. Data cleaning

Cleaning steps:

1. remove exact duplicate rows,
2. fill missing postal codes with `0`,
3. convert date columns to `datetime`,
4. standardize a few analytical fields for later use.


In [ ]:
# Keep a copy of the raw dataframe for reference
raw_df = df.copy()

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Fill Postal Code missing values
df["Postal Code"] = df["Postal Code"].fillna(0).astype(int)

# Convert date columns
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

print(f"Rows after cleaning: {df.shape[0]:,}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Unique orders: {df['Order ID'].nunique():,}")


## 5. Feature engineering

We create a few derived metrics to support the analysis:

- **Profit Margin** = `Profit / Sales`
- **Shipping Cost Ratio** = `Shipping Cost / Sales`
- **Shipping Days** = `Ship Date - Order Date`
- **Discount Band** = discount grouped into business-friendly buckets


In [ ]:
df["Profit_Margin"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"], np.nan)
df["Shipping_Cost_Ratio"] = np.where(df["Sales"] != 0, df["Shipping Cost"] / df["Sales"], np.nan)
df["Shipping_Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

discount_bins = [-0.1, 0, 0.1, 0.2, 0.3, 1.0]
discount_labels = ["No Discount", "0-10%", "10-20%", "20-30%", "30%+"]

df["Discount_Band"] = pd.cut(
    df["Discount"],
    bins=discount_bins,
    labels=discount_labels,
    include_lowest=True
)

df[["Sales", "Profit", "Profit_Margin", "Shipping_Cost_Ratio", "Shipping_Days", "Discount_Band"]].head()


## 6. Dataset overview

Quick summary of the business footprint.


In [ ]:
overview = pd.DataFrame({
    "Metric": [
        "Transaction rows",
        "Unique orders",
        "Unique customers",
        "Countries",
        "Markets",
        "Date range start",
        "Date range end"
    ],
    "Value": [
        f"{len(df):,}",
        f"{df['Order ID'].nunique():,}",
        f"{df['Customer ID'].nunique():,}",
        f"{df['Country'].nunique():,}",
        f"{df['Market'].nunique():,}",
        df["Order Date"].min().date(),
        df["Order Date"].max().date()
    ]
})

overview


## 7. Regional profitability

This view helps answer a simple but powerful question:

**Which regions generate revenue but fail to convert it into profit?**


In [ ]:
regional_profit = (
    df.groupby("Region", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Shipping_Cost=("Shipping Cost", "sum"),
          Orders=("Order ID", "nunique")
      )
)

regional_profit["Profit_Margin"] = regional_profit["Profit"] / regional_profit["Sales"]
regional_profit["Shipping_Cost_Ratio"] = regional_profit["Shipping_Cost"] / regional_profit["Sales"]

regional_profit.sort_values("Profit", ascending=False)


In [ ]:
plot_df = regional_profit.sort_values("Profit", ascending=False)

sns.barplot(data=plot_df, x="Region", y="Profit")
plt.title("Profit Contribution by Region")
plt.xlabel("Region")
plt.ylabel("Total Profit")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Regional takeaways

A useful twist here is not just looking at profit, but comparing it with **shipping cost ratio**.  
That often reveals why a region can sell a lot and still feel financially underwhelming.


In [ ]:
regional_profit.sort_values(["Profit", "Shipping_Cost_Ratio"], ascending=[True, False]).head(8)


## 8. Discount impact analysis

This is one of the most decision-friendly sections of the notebook.

Instead of analyzing discount as a messy continuous variable, we group it into ranges:

- No Discount
- 0–10%
- 10–20%
- 20–30%
- 30%+

That makes the margin story much easier to explain to stakeholders without statistical smoke machines.


In [ ]:
discount_margin = (
    df.groupby("Discount_Band", observed=False)
      .agg(
          Avg_Profit_Margin=("Profit_Margin", "mean"),
          Total_Profit=("Profit", "sum"),
          Orders=("Order ID", "nunique"),
          Rows=("Order ID", "size")
      )
      .reset_index()
)

discount_margin


In [ ]:
sns.barplot(data=discount_margin, x="Discount_Band", y="Avg_Profit_Margin")
plt.title("Average Profit Margin by Discount Band")
plt.xlabel("Discount Band")
plt.ylabel("Average Profit Margin")
plt.tight_layout()
plt.show()


### Interpretation

The key point is not that **every** order above 20% discount loses money.  
The more defensible claim is this:

> Discounts above 20% produce **negative margins on average**.

That phrasing is statistically cleaner and much harder for a reviewer to poke holes in.


## 9. Sales vs. profit by segment

A scatter plot helps us inspect volatility, outliers, and whether certain segments generate more loss-making orders.


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="Sales", y="Profit", hue="Segment", alpha=0.7)
plt.title("Sales vs Profit by Segment")
plt.xlabel("Sales")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()


## 10. Segment profitability

Now we move from individual orders to strategic customer segments.


In [ ]:
segment_summary = (
    df.groupby("Segment")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Customers=("Customer ID", "nunique"),
          Orders=("Order ID", "nunique")
      )
)

segment_summary["Profit_per_Customer"] = segment_summary["Profit"] / segment_summary["Customers"]
segment_summary["Profit_Share"] = segment_summary["Profit"] / segment_summary["Profit"].sum()

segment_summary.sort_values("Profit", ascending=False)


In [ ]:
segment_plot = segment_summary.reset_index().sort_values("Profit", ascending=False)

sns.barplot(data=segment_plot, x="Segment", y="Profit")
plt.title("Total Profit by Segment")
plt.xlabel("Segment")
plt.ylabel("Total Profit")
plt.tight_layout()
plt.show()


## 11. Category and sub-category profitability

This section identifies which product families deserve more commercial attention.


In [ ]:
category_summary = (
    df.groupby("Category")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
)

category_summary["Profit_Margin"] = category_summary["Profit"] / category_summary["Sales"]
category_summary.sort_values("Profit_Margin", ascending=False)


In [ ]:
subcategory_summary = (
    df.groupby("Sub-Category")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
)

subcategory_summary["Profit_Margin"] = subcategory_summary["Profit"] / subcategory_summary["Sales"]

subcategory_summary.sort_values("Profit_Margin", ascending=False).head(10)


In [ ]:
top_subcats = (
    subcategory_summary.sort_values("Profit", ascending=False)
    .head(10)
    .reset_index()
)

sns.barplot(data=top_subcats, x="Sub-Category", y="Profit")
plt.title("Top 10 Sub-Categories by Profit")
plt.xlabel("Sub-Category")
plt.ylabel("Total Profit")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 12. Export lightweight JSON for frontend use

This is optional for analysis, but useful if you want your React / Recharts portfolio site to consume pre-aggregated data from Python.


In [ ]:
top_regions = (
    regional_profit.sort_values("Profit", ascending=False)
    .head(4)[["Region", "Profit"]]
)

bottom_regions = (
    regional_profit.sort_values("Profit", ascending=True)
    .head(3)[["Region", "Profit"]]
)

regional_json_df = pd.concat([top_regions, bottom_regions], ignore_index=True)
regional_data = (
    regional_json_df.rename(columns={"Region": "region", "Profit": "profit"})
    .to_dict(orient="records")
)

discount_data = (
    discount_margin.rename(columns={"Discount_Band": "band", "Avg_Profit_Margin": "margin"})
    [["band", "margin"]]
    .assign(margin=lambda x: x["margin"].round(4))
    .to_dict(orient="records")
)

frontend_payload = {
    "regional_data": regional_data,
    "discount_data": discount_data
}

print(json.dumps(frontend_payload, indent=2, default=str))


## 13. Executive findings

Based on the dataset used in this notebook, the strongest portfolio-ready findings are:

1. **High discounts are margin killers.**  
   Orders discounted above 20% produce negative margins on average, especially in the 30%+ band.

2. **Consumer leads in total profit, but segment economics deserve nuance.**  
   Total profit, profit per customer, and order concentration can point to slightly different strategic choices.

3. **Regional performance is uneven.**  
   Some regions show weak profits relative to sales and shipping burden, which suggests operational inefficiency rather than demand weakness.

4. **Technology and selected sub-categories are strong profit engines.**  
   They deserve tighter inventory planning and commercial focus.

That last point is classic analytics wisdom: revenue can look glamorous while margin quietly sneaks out the back door wearing your wallet.


## 14. Business recommendations

### 1) Introduce discount guardrails
Use 20% as a soft threshold for review and approval.  
Above that point, margins deteriorate sharply on average.

### 2) Revisit pricing and promotions by segment
Rather than broad discounts, test targeted offers by customer segment and basket profile.

### 3) Audit weak regions operationally
Compare shipping cost burden, order mix, and product composition in underperforming regions.

### 4) Lean into high-margin categories
Protect stock availability and pricing strategy in categories and sub-categories with strong margins.

### 5) Evolve the project
A strong next step would be:
- customer RFM segmentation,
- churn / repeat-purchase prediction,
- or market basket analysis for cross-sell opportunities.


## 15. Final note for portfolio presentation

This notebook is designed to complement a portfolio website.

A very clean message to add to your project page is:

> *The full analysis was developed in Python using Pandas, Matplotlib and Seaborn.  
> The website uses interactive charts for presentation, while this notebook contains the original analytical workflow and business reasoning.*

That sentence removes confusion instantly and makes the project look more professional, not less.
